# Homework 3 — AReM Time-Domain Feature Extraction

**DSCI 552**  
This notebook completes the submit-ready portion of Homework 3: the AReM data description and split, time-domain feature research and extraction, feature standard deviations with 90% bootstrap confidence intervals, a data-driven selection of three features, and ISLR Exercise 3.7.4. The later binary and multiclass classification instructions in the PDF are explicitly marked for Homework 4, so they are not implemented here.

**Reproducibility:** This repository includes the 88 CSV recordings under `data/arem/`, so run the notebook from the repository root; the ZIP is not required. If the extracted data folder is absent, put `activity+recognition+system+based+on+multisensor+data+fusion+arem.zip` beside the notebook (or in an `upload/` folder), and the notebook will extract it. The notebook applies the documented minor cleaning and saves the 88-row feature matrix as `arem_instance_features.csv`. The calculations use all 88 instances for the feature-extraction questions; the prescribed training/test split is recorded for later modeling.

## 1(a)–(b). Dataset and prescribed split

AReM contains seven activity folders: `bending1`, `bending2`, `cycling`, `lying`, `sitting`, `standing`, and `walking`. Each instance is a multivariate time series with six sensor channels: `avg_rss12`, `var_rss12`, `avg_rss13`, `var_rss13`, `avg_rss23`, and `var_rss23`.

The assignment's test set contains datasets 1 and 2 from each bending folder, plus datasets 1, 2, and 3 from each of the other five folders. That gives **19 test instances** and **69 training instances**, for 88 total. This split is not used to fit a classifier in this homework section; it is preserved for later modeling.

In [1]:
from pathlib import Path
from zipfile import ZipFile
from io import StringIO
import numpy as np
import pandas as pd
from scipy.stats import f_oneway
from IPython.display import display

ZIP_NAME = "activity+recognition+system+based+on+multisensor+data+fusion+arem.zip"
DATA_ROOT = Path("data/arem")
zip_candidates = [Path(ZIP_NAME), Path("upload") / ZIP_NAME]
zip_path = next((candidate for candidate in zip_candidates if candidate.exists()), None)
has_extracted_data = bool(list(DATA_ROOT.glob("*/dataset*.csv")))

# A repository checkout may already include the extracted CSV files. Only require
# and extract the ZIP when those files are not present.
if not has_extracted_data:
    if zip_path is None:
        raise FileNotFoundError(
            f"Place the extracted activity folders in {DATA_ROOT}, or put "
            f"{ZIP_NAME!r} beside this notebook (or in an 'upload' folder)."
        )
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with ZipFile(zip_path) as archive:
        archive.extractall(DATA_ROOT)

if zip_path is not None:
    print(f"Archive available: {zip_path}")
else:
    print(f"Using extracted dataset at {DATA_ROOT}")
CHANNELS = ["avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
METRICS = ["min", "max", "mean", "median", "std", "q1", "q3"]
EXPECTED_TIMES = np.arange(0, 120000, 250)  # 480 observations, every 250 ms

def read_instance(path):
    """Read one AReM instance, handling its documented minor format variations."""
    raw_text = path.read_text(encoding="utf-8")
    data_lines = [line.rstrip().rstrip(",") for line in raw_text.splitlines()
                  if line.strip() and not line.lstrip().startswith("#")]
    first_data_line = data_lines[0]
    if "," in first_data_line:
        frame = pd.read_csv(StringIO("\n".join(data_lines)), header=None)
    else:
        # bending2/dataset4.csv is whitespace-delimited instead of comma-delimited.
        frame = pd.read_csv(StringIO("\n".join(data_lines)), header=None, sep=r"\s+")
    frame = frame.dropna(axis=1, how="all")
    if frame.shape[1] != 7:
        raise ValueError(f"Expected time + six channels in {path}; got {frame.shape[1]} columns")
    frame.columns = ["time"] + CHANNELS
    frame = frame.apply(pd.to_numeric, errors="coerce")
    if frame["time"].duplicated().any():
        raise ValueError(f"Duplicate timestamps in {path}")
    source_rows = len(frame)
    frame = frame.set_index("time").reindex(EXPECTED_TIMES)
    missing_cells = int(frame[CHANNELS].isna().sum().sum())
    if missing_cells:
        # One missing timestamp is restored using linear interpolation between its neighbors.
        frame[CHANNELS] = frame[CHANNELS].interpolate(method="index", limit_area="inside")
    if frame[CHANNELS].isna().any().any():
        raise ValueError(f"Unfilled sensor values in {path}")
    frame.index.name = "time"
    return frame.reset_index(), source_rows, missing_cells

print(f"Found {len(list(DATA_ROOT.glob('*/*.csv')))} CSV instances under {DATA_ROOT}")

Using extracted dataset at data/arem
Found 88 CSV instances under data/arem


### Data checks and cleaning

All 88 CSV files are present. Three files need minor format handling: `bending2/dataset4.csv` uses spaces between values, and `cycling/dataset9.csv` plus `cycling/dataset14.csv` have a trailing comma at the last row. `sitting/dataset8.csv` is missing the reading at 13,500 ms. The parser normalizes the delimiters and trailing commas; it restores the one missing timestamp by linear interpolation between its adjacent readings. No other sensor values are changed.

In [2]:
audit = []
raw_instances = []
for path in sorted(DATA_ROOT.glob("*/*.csv")):
    activity = path.parent.name
    dataset_id = int(path.stem.replace("dataset", ""))
    raw_text = path.read_text(encoding="utf-8")
    raw_data_lines = [line for line in raw_text.splitlines()
                      if line.strip() and not line.lstrip().startswith("#")]
    had_trailing_comma = any(line.rstrip().endswith(",") for line in raw_data_lines)
    data_lines = [line.rstrip().rstrip(",") for line in raw_data_lines]
    delimiter = "comma" if "," in data_lines[0] else "space"
    series, source_rows, interpolated_cells = read_instance(path)
    is_test = (dataset_id <= 2) if activity in {"bending1", "bending2"} else (dataset_id <= 3)
    audit.append({
        "file": f"{activity}/{path.name}",
        "source_rows": source_rows,
        "rows_after_cleaning": len(series),
        "delimiter": delimiter,
        "trailing_comma_removed": had_trailing_comma,
        "interpolated_sensor_values": interpolated_cells,
    })
    raw_instances.append((activity, dataset_id, "test" if is_test else "train", series))

audit_df = pd.DataFrame(audit)
counts = pd.Series([a for a, _, _, _ in raw_instances]).value_counts().sort_index()
split_counts = pd.Series([s for _, _, s, _ in raw_instances]).value_counts().reindex(["train", "test"])
print("Instances by activity:")
display(counts.rename("instances").to_frame())
print("Split counts:")
display(split_counts.rename("instances").to_frame())
print("Files requiring a non-default parsing or data-repair step:")
display(audit_df[(audit_df.source_rows != 480) | (audit_df.delimiter != "comma") | audit_df.trailing_comma_removed | (audit_df.interpolated_sensor_values > 0)])

Instances by activity:


activity,instances
bending1,7
bending2,6
cycling,15
lying,15
sitting,15
standing,15
walking,15


Split counts:


split,instances
train,69
test,19


Files requiring a non-default parsing or data-repair step:


file,source_rows,rows_after_cleaning,delimiter,trailing_comma_removed,interpolated_sensor_values
bending2/dataset4.csv,480,480,space,False,0
cycling/dataset9.csv,480,480,comma,True,0
cycling/dataset14.csv,480,480,comma,True,0
sitting/dataset8.csv,479,480,comma,False,6


## 1(c)(i). Common time-domain features

Common time-domain features summarize a signal directly in its observed time order, without transforming it to the frequency domain. Examples include:

- **Location:** mean and median.
- **Extremes and spread:** minimum, maximum, range, variance, standard deviation, and interquartile range.
- **Shape:** first and third quartiles, skewness, and kurtosis.
- **Signal dynamics:** root mean square, zero-crossing rate, slope, and autocorrelation at selected lags.

For this assignment, the required seven summaries are minimum, maximum, mean, median, sample standard deviation, first quartile, and third quartile.

**Research reference.** Deng, Runger, Tuv, and Martyanov (2013), *A Time Series Forest for Classification and Feature Extraction*, describe time-series classification features including mean, standard deviation (deviation), and slope, and discuss extracting these summaries over time intervals. This supports the use of simple, interpretable time-domain summaries. The assignment specifically requires the seven statistics listed above. [Paper (arXiv)](https://arxiv.org/abs/1302.2277).

## 1(c)(ii). Extract seven summaries from each of six channels

For each instance and each channel, calculate the seven specified summaries across that channel's 480 time points. This produces **6 × 7 = 42 numeric features per instance**, plus activity and split labels. The standard deviation in this step describes variation *within that instance's time series*. It is distinct from Question 1(c)(iii), which calculates the standard deviation of each feature *across the 88 instances*. Quartiles use the 25th and 75th percentiles; standard deviation uses the sample convention (`ddof=1`).

In [3]:
feature_rows = []
for activity, dataset_id, split, series in raw_instances:
    row = {"activity": activity, "dataset_id": dataset_id, "split": split}
    for channel in CHANNELS:
        values = series[channel]
        row.update({
            f"{channel}_min": values.min(),
            f"{channel}_max": values.max(),
            f"{channel}_mean": values.mean(),
            f"{channel}_median": values.median(),
            f"{channel}_std": values.std(ddof=1),
            f"{channel}_q1": values.quantile(0.25),
            f"{channel}_q3": values.quantile(0.75),
        })
    feature_rows.append(row)
feature_table = pd.DataFrame(feature_rows).sort_values(["activity", "dataset_id"]).reset_index(drop=True)
feature_columns = [f"{channel}_{metric}" for channel in CHANNELS for metric in METRICS]
assert feature_table.shape == (88, 45)  # 42 features + activity, dataset ID, split
assert feature_table[feature_columns].notna().all().all()
feature_table.to_csv("arem_instance_features.csv", index=False)
print(f"Feature table: {feature_table.shape[0]} instances × {len(feature_columns)} numeric features")
display(feature_table.head())

Feature table: 88 instances × 42 numeric features


avg_rss12_min,avg_rss12_max,avg_rss12_mean,avg_rss12_median,avg_rss12_std,avg_rss12_q1,avg_rss12_q3,var_rss12_min,var_rss12_max,var_rss12_mean,var_rss12_median,var_rss12_std,var_rss12_q1,var_rss12_q3,avg_rss13_min,avg_rss13_max,avg_rss13_mean,avg_rss13_median,avg_rss13_std,avg_rss13_q1,avg_rss13_q3,var_rss13_min,var_rss13_max,var_rss13_mean,var_rss13_median,var_rss13_std,var_rss13_q1,var_rss13_q3,avg_rss23_min,avg_rss23_max,avg_rss23_mean,avg_rss23_median,avg_rss23_std,avg_rss23_q1,avg_rss23_q3,var_rss23_min,var_rss23_max,var_rss23_mean,var_rss23_median,var_rss23_std,var_rss23_q1,var_rss23_q3,activity,dataset_id,split
37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,0.358604,0.43,0.322605,0.00,0.50,4.0,29.50,19.040937,19.250,4.462952,16.00,23.2500,0.0,7.23,0.832542,0.50,0.965659,0.00,1.120,27.25,38.25,34.311292,35.0,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30,bending1,1,test
38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,0.372437,0.47,0.289158,0.00,0.50,2.0,29.50,20.096979,21.000,3.893737,19.00,22.2500,0.0,5.76,0.876438,0.50,1.047259,0.00,1.145,27.67,38.50,33.024583,33.0,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30,bending1,2,test
35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,0.47,0.338690,0.00,0.50,6.5,29.75,22.122354,23.000,3.030943,19.75,24.0000,0.0,4.44,0.497312,0.43,0.550657,0.00,0.830,29.00,38.50,35.588458,36.0,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94,bending1,3,train
33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,0.50,0.630860,0.00,1.12,8.5,30.00,22.183625,23.000,3.810469,20.50,24.3725,0.0,5.15,0.989917,0.83,0.953730,0.43,1.300,20.00,38.67,33.493917,35.0,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00,bending1,4,train
33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,0.50,0.405469,0.43,0.71,3.0,28.25,19.006563,19.125,4.087107,16.50,22.0625,0.0,6.42,0.841875,0.50,0.928801,0.43,1.120,23.67,37.50,29.857083,30.0,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50,bending1,5,train


### Answer to 1(c)(ii)

The resulting feature matrix has **88 rows and 42 numeric feature columns** (plus three bookkeeping columns for activity, original dataset number, and train/test assignment). Each row represents one recorded activity instance; each feature column gives one summary of one sensor channel for that instance. The complete table is saved to `arem_instance_features.csv` when this cell runs.

## 1(c)(iii). Standard deviations and 90% bootstrap confidence intervals

For each of the 42 feature columns, first estimate its standard deviation across the 88 instances using the sample standard deviation. Then bootstrap the **instances**: draw 88 rows with replacement, calculate each feature's sample standard deviation, and repeat 20,000 times. The 5th and 95th percentiles of the bootstrap standard deviations form the requested two-sided 90% percentile interval. The fixed seed makes the results reproducible.

In [4]:
N_BOOT = 20_000
rng = np.random.default_rng(552)
X = feature_table[feature_columns].to_numpy(dtype=float)
feature_sd = X.std(axis=0, ddof=1)
boot_sds = np.empty((N_BOOT, X.shape[1]))
# Chunking limits temporary memory while retaining the same bootstrap procedure.
for start in range(0, N_BOOT, 1_000):
    stop = min(start + 1_000, N_BOOT)
    sample_indices = rng.integers(0, X.shape[0], size=(stop - start, X.shape[0]))
    boot_sds[start:stop] = X[sample_indices].std(axis=1, ddof=1)
ci_low, ci_high = np.quantile(boot_sds, [0.05, 0.95], axis=0)
uncertainty = pd.DataFrame({
    "feature": feature_columns,
    "sample_sd_across_instances": feature_sd,
    "90%_bootstrap_CI_lower": ci_low,
    "90%_bootstrap_CI_upper": ci_high,
})
print("Each SD is computed across the 88 instance-level values for that feature.")
display(uncertainty.round(4))

Each SD is computed across the 88 instance-level values for that feature.


feature,sample_sd_across_instances,90%_bootstrap_CI_lower,90%_bootstrap_CI_upper
avg_rss12_min,9.5700,8.2703,10.7584
avg_rss12_max,4.3944,3.3219,5.2949
avg_rss12_mean,5.3357,4.7079,5.8811
avg_rss12_median,5.4401,4.7938,5.9961
avg_rss12_std,1.7722,1.5713,1.9516
avg_rss12_q1,6.1536,5.5821,6.6454
avg_rss12_q3,5.1389,4.3319,5.8407
var_rss12_min,0.0000,0.0000,0.0000
var_rss12_max,5.0627,4.6250,5.4070
var_rss12_mean,1.5742,1.3941,1.7069


### Answer to 1(c)(iii)

The table above reports all 42 estimated standard deviations and their 90% bootstrap confidence intervals. For example, `avg_rss12_min` has an across-instance SD of about **9.5700**, with a 90% percentile bootstrap interval of **[8.2703, 10.7584]**. The interval describes uncertainty in the population SD under the instance-resampling assumption; it is not a range containing 90% of individual observations.

## 1(c)(iv). Select three important features

To make “important” concrete for this feature-extraction assignment, rank all 42 features by a one-way ANOVA **F statistic across the seven activity labels**. A high F statistic means that activity-group means differ substantially relative to variation within the groups. This is a descriptive screening criterion, not a fitted classifier or a claim of causal importance. Because this is only feature extraction, the ranking uses all 88 instances; any future predictive evaluation must perform feature selection using training data only.

In [5]:
importance_rows = []
for column in feature_columns:
    activity_groups = [feature_table.loc[feature_table.activity == activity, column].to_numpy()
                       for activity in sorted(feature_table.activity.unique())]
    f_stat, p_value = f_oneway(*activity_groups)
    importance_rows.append({"feature": column, "anova_F": f_stat, "p_value": p_value})
importance = pd.DataFrame(importance_rows).sort_values("anova_F", ascending=False).reset_index(drop=True)
print("Top 10 features by between-activity separation:")
display(importance.head(10).round(6))

Top 10 features by between-activity separation:


feature,anova_F,p_value
var_rss13_median,1012.029961,0.0
var_rss12_mean,918.613259,0.0
var_rss23_q3,860.283987,0.0
var_rss13_mean,842.094494,0.0
var_rss12_q3,819.386877,0.0
var_rss23_mean,815.955665,0.0
var_rss23_median,743.316200,0.0
var_rss13_q3,702.845559,0.0
var_rss12_median,564.354086,0.0
var_rss12_std,461.985918,0.0


### Answer to 1(c)(iv)

Using the stated criterion, select:

1. **`var_rss13_median`** — highest F statistic (**1012.03**).
2. **`var_rss12_mean`** — second highest (**918.61**).
3. **`var_rss23_q3`** — third highest (**860.28**).

These all summarize the variance/RSS channels, which vary strongly with movement intensity across activities. This is a reproducible, data-driven choice; the assignment also allows a simpler judgment-based choice such as min, mean, and max.

## 2. ISLR Exercise 3.7.4 — Training versus test RSS

The exercise compares a linear regression with a cubic regression using a single predictor and 100 observations. A cubic model contains the linear model as a special case (`β₂ = β₃ = 0`), so the models are nested. This fact determines the training-RSS comparison; test RSS also depends on the true relationship and the model's variance.

### (a) True relationship is linear: training RSS

The cubic model's training RSS must be **no larger** than the linear model's. It can be lower because the cubic terms can fit sample noise; it can equal the linear model's if the fitted quadratic and cubic coefficients add no improvement. The key is that adding terms cannot worsen the least-squares fit on the training data.

### (b) True relationship is linear: test RSS

We would generally expect the **linear model** to have lower test RSS. The true curve is linear, so the extra cubic flexibility is unnecessary and can increase variance by fitting noise. A particular finite test sample can still produce the opposite ordering by chance.

### (c) True relationship is nonlinear: training RSS

The cubic model's training RSS is again **no larger** than the linear model's, because it contains the linear model as a special case. It will be lower when the added terms improve the in-sample fit, and may be equal if they do not.

### (d) True relationship is nonlinear: test RSS

There is **not enough information to know which model will have lower test RSS** without knowing the curve's shape and the noise level. A cubic model may do better when the nonlinear pattern is well represented by a cubic. A linear model may generalize better when the curvature is mild or a cubic fit adds too much variance.

## Reproducibility and interpretation notes

- Feature standard deviations and their bootstrap intervals are computed across **instances**, after the within-instance time-series summaries are extracted.
- The bootstrap uses 20,000 resamples of 88 instances with replacement and NumPy seed 552; reported interval endpoints may vary slightly with a different seed or number of resamples.
- One missing sensor reading in `sitting/dataset8.csv` was linearly interpolated at the 13,500 ms timestamp. The two delimiter/trailing-comma cases were formatting fixes only.
- The notebook stores executed output so tables are visible in VS Code and on GitHub after saving. Re-run all cells if the data or code changes, then save the notebook.